# Paper 2 Exploration: Malthusian Dynamics


## Setup


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# Add project root to path
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

from analysis.paper2_malthus.regressions import run_fe_regression, run_rolling_window
from analysis.paper2_malthus.breaks import detect_breaks_all_entities

PANEL_PATH = os.path.join(os.path.dirname(os.getcwd()), 'data', 'country_analysis_panel.parquet')
panel = pd.read_parquet(PANEL_PATH)
print(f'Panel shape: {panel.shape}')
print(panel.dtypes)
panel.head()


## Core Malthusian Regression


In [ ]:
dep_var = 'pop_growth_rate'
indep_vars = ['popdens_p_km2_mean', 'ag_output_proxy_mha']

# Check required columns exist
available = [c for c in [dep_var] + indep_vars if c in panel.columns]
print(f'Available target columns: {available}')
print(f'All panel columns: {list(panel.columns)}')

if all(c in panel.columns for c in [dep_var] + indep_vars):
    result = run_fe_regression(panel, dep_var=dep_var, indep_vars=indep_vars, entity_col='country')
    print(f'\nFE Regression: R-squared={result["rsquared"]:.4f}, N={result["nobs"]}')
    print('\nCoefficients:')
    for var, coef in result['params'].items():
        pval = result['pvalues'][var]
        print(f'  {var}: {coef:.6f}  (p={pval:.4f})')
    print('\nFull summary:')
    print(result['summary'])
else:
    missing = [c for c in [dep_var] + indep_vars if c not in panel.columns]
    print(f'Missing columns: {missing}. Adjust dep_var/indep_vars above to match panel columns.')


## Rolling Window Estimation


In [ ]:
key_var = 'popdens_p_km2_mean'
control_vars = ['ag_output_proxy_mha']

if all(c in panel.columns for c in [dep_var, key_var] + control_vars):
    rolling_df = run_rolling_window(
        panel,
        dep_var=dep_var,
        key_var=key_var,
        control_vars=control_vars,
        entity_col='country',
        window_years=400,
        step_years=100,
    )
    print(f'Rolling window results: {len(rolling_df)} windows')
    print(rolling_df.head(10))

    if not rolling_df.empty:
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.plot(rolling_df['center_year'], rolling_df['coefficient'], marker='o', label=f'Coef: {key_var}')
        sig = rolling_df[rolling_df['pvalue'] < 0.05]
        ax.scatter(sig['center_year'], sig['coefficient'], color='red', zorder=5, label='p<0.05')
        ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
        ax.set_xlabel('Center year of window')
        ax.set_ylabel('Coefficient')
        ax.set_title(f'Rolling window: effect of {key_var} on {dep_var}')
        ax.legend()
        plt.tight_layout()
        plt.show()
    else:
        print('No rolling window results to plot (check window_years vs data range).')
else:
    missing = [c for c in [dep_var, key_var] + control_vars if c not in panel.columns]
    print(f'Missing columns: {missing}. Adjust variable names to match panel columns.')


## Structural Breaks


In [ ]:
break_dep_var = 'pop_growth_rate'
break_key_var = 'popdens_p_km2_mean'

# detect_structural_breaks uses 'popdens_lag' by default; we pass the available column name
if all(c in panel.columns for c in [break_dep_var, break_key_var]):
    breaks_df = detect_breaks_all_entities(
        panel,
        entity_col='country',
        dep_var=break_dep_var,
        key_var=break_key_var,
        min_segment=3,
        significance=0.05,
    )
    print(f'Structural breaks found: {len(breaks_df)}')
    if not breaks_df.empty:
        print('\nBreaks summary (top by F-stat):')
        print(breaks_df.sort_values('f_stat', ascending=False).head(20).to_string(index=False))
        print(f'\nCountries with breaks: {breaks_df["country"].nunique()}')
        print('\nBreak year distribution:')
        print(breaks_df['break_year'].describe())
    else:
        print('No significant structural breaks detected at p<0.05.')
else:
    missing = [c for c in [break_dep_var, break_key_var] if c not in panel.columns]
    print(f'Missing columns: {missing}. Adjust variable names to match panel columns.')
